In [1]:
import os
import json
import sqlite3
import chromadb
import uuid
from logging import exception

#from chromadb.experimental.density_relevance import collection
from groq import Groq
from google import genai
from google.genai import types
import requests
from numpy.ma.core import empty
from simpleeval import simple_eval
from ddgs import DDGS


In [2]:

gemini_api_key = os.environ.get("gemini_api_key")

client = genai.Client(api_key=gemini_api_key)

In [3]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [4]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [5]:
def get_weather_withAPI(location: str) -> str:
    """
    Gets the current weather information for a specific city or location.
    Use this tool when the user asks about the weather.
    """



    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [6]:
def calculate(answer: str)-> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool for any math calculations (e.g., '5 + 5 * 2').
    """



    try:
        answer = answer.strip()
        result = simple_eval(answer)
        return f"Calculated answer: {result}"

    except Exception as e:
        return f"Calculation Error: {str(e)}"



In [7]:
def search_web(query: str)-> str:
    """
    Searches the internet for up-to-date information, news, or facts that the agent doesn't know.
    """


    try:
        results =  DDGS().text(query,max_results= 3 )
        if len(results)==0:
             return f"Search engine error: No results found for '{query}'"


        text = ""
        for result in results:
            title = result["title"]
            body = result["body"]
            text  += f"Title: {title}\nbody: {body}\n\n---\n\n"
        return text



    except Exception as e:
        return f"Seach engine error : {str(e)}"







In [8]:
def save_memory(memory_text: str) -> str:
    """
    Saves important user information, preferences, or memories to the database.
    Use this WHENEVER the user tells you something about themselves (e.g., name, age, likes, passwords) to remember for later.
    """
    try:
        # Anı için rastgele benzersiz bir ID oluşturuyoruz
        doc_id = str(uuid.uuid4())

        # Ajanın cümlesini ChromaDB'ye kaydediyoruz
        collection.add(
            documents=[memory_text],
            ids=[doc_id]
        )
        return f"Successfully saved to memory: '{memory_text}'"
    except Exception as e:
        return f"A problem occurred while saving: {str(e)}"

In [9]:
def get_user_profile(key: str) -> str:
    """
    Retrieves specific, structured profile information about the user (e.g., name, age, email, city).
    Do NOT use this for past memories, passwords, or favorite items. Use search_vector_database for those.
    """

    profile_db = {"name": "Ahmet", "city": "Istanbul"}
    return profile_db.get(key, "No data found for this key.")

In [28]:
def chunk_text(text:str, chunks_size: int = 1000,overlap:int = 200 )-> list:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunks_size
        chunks.append(text[start:end])
        start += chunks_size - overlap
    return chunks



In [27]:
def process_long_documents(file_path:str)-> str:
    """a func that only you can use to embed long documents into database"""

    try:
        with open(file_path,"r",encoding="utf-8") as file:
            full_text = file.read()

        chunks = chunk_text(full_text,chunks_size= 1000,overlap= 200 )

        ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

        collection.add(documents=chunks, ids=ids)
        return f"Successfully processed {len(chunks)} long documents."
    except Exception as e:
        return f"Error occured while reading the file: {str(e)}"


In [12]:
def search_vector_database(query: str) -> str:
    """
    Searches the user's personal documents, past notes, memories, and general knowledge database.
    ALWAYS use this tool to find information about past events (like what the user ate), passwords,
    favorite items (consoles, games, food), or any saved free-text memory.
    """
    try:
        collec = collection.query(query_texts=[query],n_results=3)
        if len(collec["documents"][0]) > 0:
            found_chunks = collec["documents"][0]
            combined_context = "\n\n--- connecting context ---\n\n".join(found_chunks)
            return f"The resutls for the seach:\n{combined_context}."
        else:
            return "The searched term could not be found in the database."
    except Exception as e:
        return f"Error: {str(e)}"

In [13]:
connection = sqlite3.connect("memory.db",check_same_thread=False)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
 key TEXT PRIMARY KEY,
 value TEXT)
  """)
connection.commit()

In [37]:
chroma_client = chromadb.PersistentClient(path="C:/Agents/agent_memory_db")

collection = chroma_client.get_or_create_collection(name="agentic_databank")

collection.add(
    documents=[
        "Last night I ate chiken wings at it was delicous I bought it from Lidl.",
        "The password for the office door is 2099.",
        "my favourite console is playstation 2. "
    ],
    ids=["id1","id2",",id3"]

)

print("vector dataset is loaded")

vector dataset is loaded


In [15]:
system_prompt = """
You are an advanced AI assistant with access to several tools.
Think logically step-by-step to answer the user's questions.
If you need information you don't have, use the appropriate tool.
Once you have all the information you need, provide a clear, helpful final answer.
"""

In [44]:
def run_reAct_agent(user_query: str, tools: list):

    config = types.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=tools,
        temperature=0.1
    )


    chat = client.chats.create(
        model="gemini-2.5-pro",
        config=config
    )

    print("\n--- Gemini ReAct cycle started ---")
    print("[System] Agent is  thinking, choosing the tools and running them on the background...")


    response = chat.send_message(user_query)


    final_answer = response.text
    print(f"\nFinal Answer: {final_answer}")

    return final_answer

In [40]:
my_tools_real = [
    get_weather_withAPI,
    calculate,
    search_web,
    save_memory,
    search_vector_database
]

In [26]:
print("\nANSWER:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---


AttributeError: 'Client' object has no attribute 'chat'

In [59]:
print("\nANSWER:",run_reAct_agent(" what is the sqaure root of 16",tools=my_tools_real))


--- Adım 1 ---

Final Answer: The square root of 16 is 4.

ANSWER: The square root of 16 is 4.


In [60]:
print("\nANSWER:",run_reAct_agent(" what is the retail price of Gta 6 for Playtation 5",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_web
The chosen action is {"query":"GTA 6 PlayStation 5 retail price"}
The chosen parameters is {'query': 'GTA 6 PlayStation 5 retail price'}
The chosen action is Title: GTA 6 vorbestellen: Preise, Editionen & Release-Termin im Überblick
body: 2 weeks ago - GTA 6 kann ab dem 25. Juni 2026 vorbestellt werden. Alle Infos zu den Preisen, Editionen, PS5 Bundle und dem Release am 19. November 2026. Dieser Beitrag wird fortlaufend aktualisiert!

---

Title: Grand Theft Auto 6 price finally announced for US customers, and later revealed for UK and European customers.
body: 2 weeks ago - The standard version of GTA 6 will retail for $79.99, and be available on Xbox Series X/S and PlayStation 5.

---

Title: r/PS5 on Reddit: Grand Theft Auto 6 will retail for $80 and the Ultimate Edition will be $100
body: 2 weeks ago - GTA 6 price listed as $115 via one retailer as pre-order rumors circulate, and fans are praying it's j

In [61]:
print("\nANSWER:",run_reAct_agent(" my name is Ali  and my favourite game is GTA 6",tools=my_tools_real))



--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is save_memory
The chosen action is {"key":"name","value":"Ali"}
The chosen parameters is {'key': 'name', 'value': 'Ali'}
The chosen action is Successfully saved name and Ali in memory.

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is save_memory
The chosen action is {"key":"favourite_game","value":"GTA 6"}
The chosen parameters is {'key': 'favourite_game', 'value': 'GTA 6'}
The chosen action is Successfully saved favourite_game and GTA 6 in memory.

--- Adım 3 ---

Final Answer: Got it, Ali! I’ve saved your name and noted that your favorite game is GTA 6. If there’s anything else you’d like to share or ask about, just let me know!

ANSWER: Got it, Ali! I’ve saved your name and noted that your favorite game is GTA 6. If there’s anything else you’d like to share or ask about, just let me know!


In [62]:
print("\nANSWER:",run_reAct_agent(" what is my name and my favourite game ",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"name"}
The chosen parameters is {'key': 'name'}
The chosen action is Data found in memory: name = Ali

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"favourite game"}
The chosen parameters is {'key': 'favourite game'}
The chosen action is Data found in memory: favourite game = GTA 6

--- Adım 3 ---

Final Answer: Your name is **Ali**, and your favorite game is **GTA 6**.

ANSWER: Your name is **Ali**, and your favorite game is **GTA 6**.


In [21]:
print("\nANSWER:",run_reAct_agent("look through my files and tell me what I did my files name is yesterday  and tell me what I did ",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_my_documents
The chosen action is {"query":"yesterday"}
The chosen parameters is {'query': 'yesterday'}
The chosen action is An error occurred: [Errno 2] No such file or directory: 'C:/Agents/26-07-07.txt'

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is search_my_documents
The chosen action is {"query":"yesterday.txt"}
The chosen parameters is {'query': 'yesterday.txt'}
The chosen action is An error occurred: [Errno 2] No such file or directory: 'C:/Agents/26-07-07.txt'

--- Adım 3 ---

Final Answer: I’m sorry, but I wasn’t able to locate a file named **“yesterday”** (or any similarly‑named document) in your stored files, so I can’t retrieve its contents for you. If you can provide the exact filename (including its extension) or upload the file again, I’ll be happy to look through it and let you know what you wrote about.

ANSWER: I’m sorry, but I wasn’t able to locate a file named **“yeste

In [27]:
print("\nANSWER:",run_reAct_agent("what did I ate last night  ?",tools=my_tools_real))

ValidationError: 1 validation error for GenerateContentConfig
tools
  Input should be a valid list [type=list_type, input_value={'get_weather': <function... at 0x00000279BA23D260>}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type

In [38]:

api_tools = my_tools_real

# Ajanı çalıştır
answer = run_reAct_agent("what is my name and my fav game   ?", tools=api_tools)


--- Gemini ReAct cycle started ---
[System] Agent is  thinking, choosing the tools and running them on the background...

Final Answer: Your name is Ali, and your favorite game is GTA 6!


In [45]:
print(process_long_documents("C:/Agents/book.txt"))

Successfully processed 2 long documents.


In [46]:
# Test 1: Gizli bilgi sorgusu
answer1 = run_reAct_agent("What is the password to open Dr. Thorne's safe?", tools=my_tools_real)

# Test 2: Sebep-sonuç sorgusu
answer2 = run_reAct_agent("Why did the miners' eyes turn purple on Zephyria?", tools=my_tools_real)

# Test 3: Çıkarım sorgusu (Farklı parçalardaki bilgiyi birleştirme)
answer3 = run_reAct_agent("If I want to find the Omega Alliance files, who should I talk to and where should I go?", tools=my_tools_real)


--- Gemini ReAct cycle started ---
[System] Agent is  thinking, choosing the tools and running them on the background...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 42.537550487s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': 